# Next step: step 4b, centered heat
**What it adds:** `heat_mode`. Step 4a showed Wallin's heat kicks are all positive, so a heated disk drifts toward
+x, +y, +z by about heat/2. The fix keeps Wallin's version and adds a centered one:

| `heat_mode` | Kick added to each of vx, vy, vz | Mean | Spread |
|---|---|---|---|
| `"centered"` (default in `sim`) | in [−heat/2, heat/2) | 0: no drift | width heat |
| `"wallin"` (the original) | in [0, heat) | heat/2: a drift | width heat, the same |

Only the mean changes, so a given `heat` adds the same amount of random motion in both modes, and runs compare
cleanly. `sim` uses `"centered"` by default. `jspam` keeps `"wallin"` as its default, so the fork on its own
still reproduces the original program (`PLAN.md`, Physics fixes).

**How it connects:**
- **Below, in JSPAM:** the first edit to Wallin's physics code, one line in `profile` (×3 for vx, vy, vz).
- **Next (4c):** `SimulatorConfig` bundles `n1`, `n2`, `potential`, `heat1`, `heat2` and `heat_mode`.

The full, current demo is `simulator_tour.ipynb`.

**To run it** (from this repo's root, in bash or PowerShell), then open the `127.0.0.1` link it prints:
```bash
docker run --rm -it -p 8888:8888 -v "${PWD}:/gtc" -w /gtc -e UV_PROJECT_ENVIRONMENT=/tmp/venv \
    ogdenm12/gtc-base-image:v2 uv run --group notebook jupyter lab --ip=0.0.0.0 --allow-root --no-browser
```
- `-v "${PWD}:/gtc"` mounts this repo into the container, so edits on Windows show up immediately.
- `UV_PROJECT_ENVIRONMENT=/tmp/venv` keeps the Linux virtual environment inside the container, out of the repo.
  It's rebuilt each time the container starts (a minute or two, mostly compiling SPAM).
- Our package is installed in editable mode. After editing a `.py` file, restart the kernel to pick it up.

## What changed
**JSPAM, `fortran/parameters_module.f90`:** a new module variable.
```fortran
! SIMR: subtracted from randm() when heat is added (see profile in setup_module).
! 0 is Wallin's original (kicks in [0, heat), so a heated disk drifts); 0.5 centers them.
real (kind=8) :: heat_offset = 0.0d0
```
**JSPAM, `fortran/setup_module.f90`, Wallin's `profile`:** the only physics edit. With `heat_offset = 0`, `(randm() - 0)`
is exactly `randm()`, so the default gives bit-identical results to the original.
```fortran
! before
x0(i,4) = xv3  + randm()*heat
! after (the same for x0(i,5) and x0(i,6))
x0(i,4) = xv3  + (randm() - heat_offset)*heat
```
**JSPAM, `fortran/simr_interface.f90`:** `run_collision`, `initial_state` and `setup_run` take `heat_mode`, and
`setup_run` sets the offset every run (so one run's mode can't leak into the next):
```fortran
if (heat_mode == 1) then
  heat_offset = 0.5d0
else
  heat_offset = 0.0d0
end if
```
**JSPAM, `src/jspam/__init__.py`:** `HEAT_MODES = {"wallin": 0, "centered": 1}`, and `heat_mode="wallin"` on
`run_collision` and `initial_state`. An unknown name raises an error.

**`sim.py`:** `simulate` and `initial_state` take `heat_mode="centered"` and pass it on.

**`tests/test_sim.py`, new:** tests for everything in `sim` so far, one section per step. Run them in the
container with `uv run --group dev pytest -m "not slow"`.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from galactic_tidal_constraints.simr import sim

MODEL_STRING = "-9.93853,-4.5805,15.43348,-1.54025,-2.97899,4.69234,34.41671,29.49607,2.97913,4.2501,301.0,35.5,310.80745,321.98757,0.3,0.3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0,0.24008,-2.41999,5.62715,0.0,1.0,0.0,1.0,0.0,0.0"
params = sim.parse_model_string(MODEL_STRING)
N1, N2 = 2000, 2000
SEED = 1
HEAT = 0.5
MODES = ["wallin", "centered"]

## 1. The kicks, both modes
Same seed throughout, so the two modes use the same random numbers. Subtracting a cold start isolates the kick.

In [ ]:
cold = sim.initial_state(params, n1=N1, n2=N2, seed=SEED)
kicks = {}
for mode in MODES:
    hot = sim.initial_state(params, n1=N1, n2=N2, seed=SEED, heat1=HEAT, heat2=HEAT, heat_mode=mode)
    kicks[mode] = hot[:-1, 3:] - cold[:-1, 3:]
    k = kicks[mode]
    print(f"{mode:8s}  min {k.min():+.3f}   max {k.max():+.3f}   mean per axis {k.mean(axis=0).round(3)}"
          f"   spread (max - min) {k.max() - k.min():.3f}")
print("the two modes differ by exactly heat/2:", np.allclose(kicks["wallin"] - kicks["centered"], HEAT / 2))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for mode in MODES:
    ax.hist(kicks[mode].ravel(), bins=60, histtype="step", label=mode)
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("velocity added by heat (vx, vy and vz together)")
ax.set_ylabel("particles")
ax.legend()
plt.show()

## 2. Each disk's net velocity
Mean velocity of each disk's particles relative to its galaxy's center. Wallin's: about heat/2 per axis.
Centered: about 0, like no heat at all.

In [ ]:
def disk_mean_velocity(state, first, last):
    """Mean (vx, vy, vz) of a disk's particles, relative to its galaxy's center."""
    center = np.zeros(3) if first == 0 else state[-1, 3:]  # the primary's center is at rest at the origin
    return (state[first:last, 3:] - center).mean(axis=0)

print(f"{'no heat':9s} primary: {disk_mean_velocity(cold, 0, N1).round(3)}   "
      f"secondary: {disk_mean_velocity(cold, N1, N1 + N2).round(3)}")
for mode in MODES:
    hot = sim.initial_state(params, n1=N1, n2=N2, seed=SEED, heat1=HEAT, heat2=HEAT, heat_mode=mode)
    print(f"{mode:9s} primary: {disk_mean_velocity(hot, 0, N1).round(3)}   "
          f"secondary: {disk_mean_velocity(hot, N1, N1 + N2).round(3)}")

## 3. By the end of the run
The primary disk's mean position, which sits near its center (the origin) without heat. Wallin's heat carries
the disk off toward +x, +y, +z; centered heat should keep it close to the no-heat run while still puffing the disk up.

In [ ]:
runs = {"no heat": sim.simulate(params, n1=N1, n2=N2, seed=SEED)}
for mode in MODES:
    runs[mode] = sim.simulate(params, n1=N1, n2=N2, seed=SEED, heat1=HEAT, heat2=HEAT, heat_mode=mode)

for title, state in runs.items():
    print(f"{title:9s} primary disk's mean position: {state[:N1, :3].mean(axis=0).round(3)}")

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)
for ax, (title, state) in zip(axes, runs.items()):
    ax.scatter(state[:N1, 0], state[:N1, 1], s=1)
    ax.scatter(state[N1:N1 + N2, 0], state[N1:N1 + N2, 1], s=1)
    ax.plot(0, 0, "k+", ms=14)
    ax.plot(*state[:N1, :2].mean(axis=0), "ro", ms=6, label="primary disk's mean position")
    ax.set_title(title if title == "no heat" else f"{title}, heat {HEAT}")
    ax.set_aspect("equal")
axes[0].legend()
plt.show()

## 4. A typo is caught

In [ ]:
try:
    sim.simulate(params, n1=10, n2=10, heat1=HEAT, heat_mode="centred")
except ValueError as error:
    print(error)